# Advanced 12 — Agent Benchmarks & Enterprise Evals

Public benchmarks provide bounded capability evidence. In this lab you build a versioned enterprise evaluation system that governs cases, validates observable trajectories, separates harness failures from agent failures, compares a baseline and candidate on the same cases, and applies risk-specific release policy.

**Authority boundary:** the agent produces observations; the application-owned harness validates outcomes, evidence, policy, metrics, and release.

## 1. Load the tested implementation

The notebook imports the same `policy.py` and `lab.py` used by pytest. It requires no model credentials or network calls. Frozen profiles validate benchmark plumbing and policy behavior—not real model intelligence or generalization.

In [ ]:
from pathlib import Path
import sys

course_dir = Path.cwd()
if not (course_dir / 'lab.py').exists():
    course_dir = Path('curriculum/advanced/12-agent-benchmarks')
sys.path.insert(0, str(course_dir.resolve()))

from lab import (
    BENCHMARK_ID, DATASET_VERSION, DEFAULT_RELEASE_POLICY,
    benchmark_metrics, benchmark_suite, evaluate_case, evaluate_profile,
    fixture_observation, northstar_cases, paired_comparison,
    release_decision, run_manifest, sanitize_trace, sensitive_findings,
    suite_report, wilson_interval,
)
from policy import CaseOutcome, DatasetSplit

suite = benchmark_suite()
{'benchmark_id': BENCHMARK_ID, 'dataset_version': DATASET_VERSION, 'case_count': len(suite.cases), 'owners': suite.owner_groups}

The suite identity also binds environment, evaluator, and policy versions. A case has an immutable ID/version/digest plus review status. Changing expected behavior without a new reviewed version breaks the digest.

In [ ]:
cases = northstar_cases()
{
    'split_support': {split.value: sum(case.split is split for case in cases) for split in DatasetSplit},
    'provenance': sorted({case.provenance.value for case in cases}),
    'scenario_tags': sorted({tag for case in cases for tag in case.scenario_tags}),
    'risk_tags': sorted({tag for case in cases for tag in case.risk_tags}),
}

## 2. Minimize and scan trace data

Production traces are valuable but biased and sensitive. They become **candidate cases** only after purpose review, minimization, structured and free-text detection, expected-behavior investigation, and human approval. A stable keyed identifier is a pseudonym—not anonymity.

In [ ]:
raw_trace = {
    'user_email': 'jane@example.com',
    'request': {'message': 'Contact jane@example.com; token tok-abcdefghijk'},
    'tenant_class': 'regulated',
}
clean_trace = sanitize_trace(raw_trace)
{'clean': clean_trace, 'remaining_findings': sensitive_findings(clean_trace)}

The simple detector is an educational control, not a production DLP system. Real trace use also needs access, retention, key management, regional/contractual constraints, and legal/compliance review.

## 3. Specify observable behavior, not hidden reasoning

Cases use required/allowed/forbidden tool sets and partial-order constraints. `required ∪ allowed` is the complete allowlist: every other tool call hard-fails, while explicitly forbidden tools add a critical classification. This accepts safe alternate orderings while enforcing material boundaries. Grounding comes from final-claim citations checked against an application-owned evidence registry—not observed metadata or an agent-authored `grounded=True` flag. `allow_abstention` is permission; the observed `AgentDecision` records whether the agent answered or abstained.

In [ ]:
case = next(case for case in cases if case.case_id == '03-safe-mitigation')
{
    'required': case.tools.required,
    'allowed': case.tools.allowed,
    'forbidden': case.tools.forbidden,
    'partial_order': [(rule.before, rule.after) for rule in case.ordering],
    'required_evidence': case.expected.required_evidence_ids,
}

## 4. Evaluate baseline and candidate on identical held-out cases

Four development cases are visible for iteration. Release evidence comes from the 16 validation/challenge cases. The environment resets between cases. Before scoring, the harness binds case → run manifest → observation across environment, fixture, tool versions, knowledge snapshot, and cache policy. Paired comparisons additionally require compatible benchmark, dataset, case, evaluator, policy, environment, and fixture versions.

In [ ]:
held_out = tuple(case for case in cases if case.split is not DatasetSplit.DEVELOPMENT)
baseline = evaluate_profile('baseline', held_out)
candidate = evaluate_profile('candidate', held_out)
comparison = paired_comparison(baseline, candidate, held_out)
{
    'improvements': comparison.improvements,
    'regressions': comparison.regressions,
    'critical_regressions': comparison.critical_regressions,
    'changed_cases': [(row.case_id, row.classification) for row in comparison.rows if row.classification != 'UNCHANGED'],
}

The candidate improves model routing, notification suppression, and restart recovery, but newly reads foreign-tenant evidence. A platform control may contain an action, yet an attempted forbidden action still fails agent-policy adherence. Containment and safe behavior are different measurements.

## 5. Report uncertainty, slices, invalid runs, cost, and latency

A percentage without support is incomplete. Binary rates use Wilson intervals, and policy may optionally gate on the lower bound rather than only the point estimate. Zero observed critical failures in a finite run does not prove a zero production failure probability. Invalid harness runs stay outside agent-quality denominators but remain visible. Release aggregation requires exactly one result per selected case unless the pre-run manifest records a governed exclusion. Wall-clock latency is not the sum of model/tool/queue work, especially under parallelism.

In [ ]:
metrics = benchmark_metrics(candidate, held_out)
{
    'valid_agent_runs': metrics.valid_agent_runs,
    'invalid_runs': metrics.invalid_runs,
    'compliant_success': metrics.compliant_success.model_dump(),
    'critical_failure': metrics.critical_failure.model_dump(),
    'cost_per_successful_compliant_task_usd': round(metrics.cost_per_successful_compliant_task_usd or 0, 4),
    'p50_wall_clock_ms': metrics.p50_wall_clock_ms,
    'p95_wall_clock_ms': metrics.p95_wall_clock_ms,
}

In [ ]:
[
    {'slice': row.slice_id, 'n': row.support, 'compliant_success': row.compliant_success_rate, 'critical_failures': row.critical_failures}
    for row in metrics.per_slice
    if row.slice_id in {'authorization', 'tenant-isolation', 'financial', 'adversarial'}
]

In [ ]:
small_perfect = wilson_interval(3, 3)
large_perfect = wilson_interval(300, 300)
{
    '3_of_3': (small_perfect.estimate, round(small_perfect.lower, 3), round(small_perfect.upper, 3)),
    '300_of_300': (large_perfect.estimate, round(large_perfect.lower, 3), round(large_perfect.upper, 3)),
}

Both estimates are 100%, but their uncertainty differs materially. Release policy also sets minimum support for critical slices. Rare high-severity cases should be intentionally represented; prevalence and importance are not the same thing.

## 6. Hard gates dominate averages

In [ ]:
decision = release_decision(metrics, comparison, DEFAULT_RELEASE_POLICY)
{
    'compliant_success_rate': metrics.compliant_success.estimate,
    'threshold': DEFAULT_RELEASE_POLICY.min_compliant_success_rate,
    'decision': decision.status.value,
    'permits_release': decision.permits_release,
    'reason_codes': decision.reason_codes,
}

The aggregate exceeds the 80% threshold, but the blocking policy still rejects the candidate. Routine improvements cannot buy off a cross-tenant regression. New or noisy metrics should begin in `SHADOW`, mature through `ADVISORY`, and become `BLOCKING` only when their meaning and response are reliable. A live, scoped exception record does not mutate this decision or itself authorize release; exception application belongs to a separate authorized workflow.

## 7. Repair and rerun the exact comparison

In [ ]:
{'candidate': suite_report('candidate'), 'repaired': suite_report('repaired')}

The repaired fixture passes the same cases, environment, metrics, and release policy. Do not change the expected answer merely to make a candidate green. Dataset, evaluator, environment, and policy changes are versioned and reviewed separately.

## 8. Failure injection: harness failure is not agent failure

In [ ]:
failed_case = held_out[0]
invalid = evaluate_case(
    failed_case, fixture_observation(failed_case, 'harness-error'),
    run_manifest('harness-error', 'run-harness-error'),
)
{'outcome': invalid.outcome.value, 'task_success': invalid.task_success, 'reasons': invalid.failure_reasons}

A broken sandbox yields `INVALID_RUN`, not an agent-quality failure. Surface environment startup, fixture, tool-simulation, timeout-origin, and evaluator health. Repair the harness and restore required valid support before drawing a release conclusion.

## Production extensions

- Add repeated trials only for stochastic/model-dependent cases; report success probability, variance, and flakiness instead of rerunning until green.
- Freeze agent outputs when an evaluator changes; run old and new evaluator versions to separate measurement drift from agent drift.
- Maintain a representative workload suite and a deliberately overrepresented safety/challenge suite; do not combine their aggregates without explaining weights.
- Extend the operational contract for multi-agent coordination, routing errors, governed memory, proactive notification quality, long-running recovery, and world-model calibration.
- Move through offline → shadow → canary → production monitoring, then curate failures and evaluator disagreements into the next reviewed dataset version.

## Exercises and checkpoint

1. Add an approval-before-write partial-order case and a changed-proposal replay failure.
2. Add repeated trials and a flaky-case report without majority-vote hiding.
3. Add workload-weighted metrics while preserving unweighted and slice reports.
4. Add an evaluator-version recomputation report over frozen outputs.

**Checkpoint:** Why does `87.5% compliant success` still block here? Because a separate zero-tolerance cross-tenant hard gate and critical-regression budget failed. Release is an application policy over criterion-level evidence, not one weighted average.